#Time Series - LAB 3

In [1]:
#import libraries
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore")

#Question 1: Design an ARIMA model for the Female Births in California database. Use the model for forecasting.

In [ ]:
# Load the dataset
data = pd.read_csv('daily-total-female-births-CA.csv')
data['date'] = pd.to_datetime(data['date'])
data.set_index('date', inplace=True)

# Plot the time series
plt.plot(data['births'])
plt.title('Daily Total Female Births in California')
plt.xlabel('Date')
plt.ylabel('Number of births')
plt.show()


In [ ]:
#Check if there is NaN in the database
print(f'Number of NaN: {data.births.isna().sum()}')

#Do statistical stationarity test
#Dickey-Fuller test
adf_test = adfuller(data.births)
# Output the results
print('ADF Statistic: %f' % adf_test[0])
print('p-value: %f' % adf_test[1])
if (adf_test[1] < 0.05): #A p-value below 0.05 indicates stationarity, and our data meets this criterion, so we do not need to difference it.
  print('The time series is stationarity.')

In [ ]:
#Plots ACF and PACF
plot_acf(data.births,lags=30) #plot acf
plot_pacf(data.births,lags=30) #plot pacf
plt.show()

##Using the model to do a long-term forecasting

In [ ]:

# Split the data into train and test
train_size = int(len(data) * 0.8)
train, test = data[0:train_size], data[train_size:len(data)]

# Fit the ARIMA model on the training dataset
model_train = ARIMA(train.births, order=(1, 0, 1), freq='D') #ARMA model (p = 1, d = 0, q = 1)
model_train_fit = model_train.fit()

# Forecast on the test dataset
test_forecast = model_train_fit.get_forecast(steps=len(test)) # Predicts all the data in the test set
test_forecast_series = pd.Series(test_forecast.predicted_mean, index=test.index)

# Create a plot to compare the forecast with the actual test data
plt.figure(figsize=(14,7))
plt.plot(train.births, label='Training Data')
plt.plot(test.births, label='Actual Data', color='orange')
plt.plot(test_forecast_series, label='Forecasted Data', color='green')
plt.fill_between(test.index,
                 test_forecast.conf_int().iloc[:, 0],
                 test_forecast.conf_int().iloc[:, 1],
                 color='k', alpha=.15)
plt.title('ARIMA Model Evaluation')
plt.xlabel('Date')
plt.ylabel('Number of Births')
plt.legend()
plt.show()

**##Performance metrics**

The R² metric, also known as the coefficient of determination, represents the percentage of the data variance that is explained by the model. The results range from 0 to 1 and are usually also expressed in percentage terms, that is, ranging from 0% to 100%. The higher the R² value, the more explanatory the model is in relation to the predicted data.

\begin{equation}
R^2(y,\hat{y}) = 1 - \frac{\sum^{n}_{i=1}{(y_i - \hat{y_i})}^2}{\sum^{n}_{i=1}{(y_i - \bar{y_i})}^2}
\end{equation}

The MSE (Mean Squared Error) is a metric that calculates the average difference between the predicted and actual values. However, the difference is squared, penalizing values ​​that are very different between the predicted and actual values.

\begin{equation}
MSE(y,\hat{y}) = \frac{1}{n}\sum^{n}_{i=1}{(y_i - \hat{y_i})}^2
\end{equation}

The RMSE (Root Mean Squared Error) is basically the same calculation as MSE, but to deal with the problem of the difference between units, the square root is applied. This way, the unit is on the same scale as the original data, resulting in better interpretability.

\begin{equation}
RMSE(y,\hat{y}) = \sqrt{\frac{1}{n}\sum^{n}_{i=1}{(y_i - \hat{y_i})}^2}
\end{equation}

The MAE (Mean Absolute Error) measures the average difference between the real value and the predicted value. However, since there are positive and negative values, a module is added between the difference in values. Furthermore, this metric is not affected by outliers as much as MSE and RMSE.

\begin{equation}
MAE(y,\hat{y}) = \frac{1}{n}\sum^{n}_{i=1}{|y_i - \hat{y_i}|}
\end{equation}

The MAPE (Mean Absolute Percentage Error) is a metric that shows the percentage of error in relation to the actual values. For example, if the MAPE result is equal to 20%, then it means that the model makes predictions that on average the difference between the predicted value and the actual value is equivalent to 20% of the actual value (either more or less). It should be avoided in situations where the actual values ​​are close to zero.

\begin{equation}
MAPE(y,\hat{y}) = \frac{1}{n}\sum^{n}_{i=1}{\frac{|y_i - \hat{y_i}|}{\max(\epsilon,|y_i|)}} \times 100\%
\end{equation}

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

# Calculate the R²
train_forecast = model_train_fit.predict(start=0, end=len(train), dynamic=False) # Predicts all the data in the train set
train_forecast_series = pd.Series(train_forecast, index=train.index)
r2 = r2_score(train.births, train_forecast_series)
print('R2:',r2)

# Calculate the mean squared error
mse = mean_squared_error(test.births, test_forecast_series)
print('MSE:', mse)

# Calculate the root mean squared error
rmse = mse**0.5
print('RMSE:', rmse)

# Calculate the mean absolute error
mae = mean_absolute_error(test.births, test_forecast_series)
print('MAE:', mae)

# Calculate the mean absolute percentage error
mape = mean_absolute_percentage_error(test.births, test_forecast_series)
print('MAPE:', 100*mape)

##Residue analysis

In [ ]:
model_train_fit.plot_diagnostics(figsize=(15, 12))
plt.show()

##Using the model to predict *n* periods into the future (n equals the forecast horizon)

In [ ]:
horizon = 1

# Split the data into train and test
train_size = int(len(data) * 0.8)
train, test = data[0:train_size], data[train_size:len(data)]

# Fit the ARIMA model on the training dataset
model_train = ARIMA(train.births, order=(1, 0, 1)) #ARMA model (p = 1, d = 0, q = 1)
model_train_fit = model_train.fit()

#dynamic (bool, optional) – The dynamic keyword affects in-sample prediction.
#If dynamic is False, then the in-sample lagged values are used for prediction.
#If dynamic is True, then in-sample forecasts are used in place of lagged dependent variables. The first forecasted value is start.

# Forecast on the test dataset
predictions_f = []
train_data = [x for x in train.births]
for t in range(len(test)-(horizon-1)):
  model_train = ARIMA(train_data, order=(1, 0, 2)) #ARMA model (p = 1, d = 0, q = 1)
  model_train_fit = model_train.fit()
  yhat_f = test_forecast = model_train_fit.predict(start=len(train_data), end=len(train_data)+horizon-1, dynamic=True)[horizon-1]
  predictions_f.append(yhat_f)
  train_data.append(test.births[t])

# Forecast on the test dataset
test = data[train_size+horizon-1:len(data)]

test_forecast_series = pd.Series(predictions_f, index=test.index)

# Create a plot to compare the forecast with the actual test data
plt.figure(figsize=(14,7))
plt.plot(train.births, label='Training Data')
plt.plot(test.births, label='Actual Data', color='orange')
plt.plot(test_forecast_series, label='Forecasted Data', color='green')
plt.title('ARIMA Model Evaluation')
plt.xlabel('Date')
plt.ylabel('Number of Births')
plt.legend()
plt.show()

# Calculate the root mean squared error
mse = mean_squared_error(test.births, test_forecast_series)
rmse = mse**0.5
print('RMSE:', rmse)

#Question 2: Use a simple neural network to predict the mean temperature in Delhi using exogenous inputs

In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import numpy as np

In [ ]:
# Load the dataset
df_train = pd.read_csv("/content/DailyDelhiClimateTrain.csv") #, index_col="date")
display(df_train)

df_test = pd.read_csv("/content/DailyDelhiClimateTest.csv") #, index_col="date")
display(df_test)

df_train['date'] = pd.to_datetime(df_train['date'])
df_train.set_index('date', inplace=True)

df_test['date'] = pd.to_datetime(df_test['date'])
df_test.set_index('date', inplace=True)

# Plot the time series
plt.plot(df_train.meantemp)
plt.title('Daily Mean Temperature in Delhi')
plt.xlabel('Date')
plt.ylabel('°C')
plt.show()

In [20]:
#Leaves variables with zero mean and unitary variance

df_val = df_train[-100:]
df_train = df_train[:-100]

scaler_type=StandardScaler()
df_train = scaler_type.fit_transform(df_train)
df_val = scaler_type.transform(df_val)
df_test = scaler_type.transform(df_test)

In [ ]:
lags = 3

train_data = []
train_output = []

for i in range((df_train.shape[0])-(lags)-1):
  tmp = []
  tmp.append(df_train[i:i+lags,:].T)
  train_data.append(np.array(tmp).ravel())
  train_output.append(df_train[i+lags+1,0])

train_data = np.array(train_data)
train_output = np.array(train_output)


val_data = []
val_output = []

for i in range((df_val.shape[0])-(lags)-1):
  tmp = []
  tmp.append(df_val[i:i+lags,:].T)
  val_data.append(np.array(tmp).ravel())
  val_output.append(df_val[i+lags+1,0])

val_data = np.array(val_data)
val_output = np.array(val_output)


test_data = []
test_output = []

for i in range((df_test.shape[0])-(lags)-1):
  tmp = []
  tmp.append(df_test[i:i+lags,:].T)
  test_data.append(np.array(tmp).ravel())
  test_output.append(df_test[i+lags+1,0])

test_data = np.array(test_data)
test_output = np.array(test_output)



print(train_data.shape)
print(train_output.shape)

print(val_data.shape)
print(val_output.shape)

print(test_data.shape)
print(test_output.shape)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(12,)),        # Input layer
    layers.Dense(32, activation='relu'),  # Hidden layer with 32 neurons and ReLU activation
    layers.Dense(1, activation='linear')  # Output layer with a single neuron (for regression)
])

model.compile(optimizer='adam', loss='mean_squared_error')
#loss=tf.keras.losses.Huber(delta=100.0)
#model.compile(optimizer='adam', loss=loss)

# You can adjust the number of epochs and batch size based on your data and resources.
model.fit(train_data, train_output, epochs=50, batch_size=32, validation_data=(val_data, val_output))


# Evaluate the model on the test data
test_loss = model.evaluate(test_data, test_output)
print(f"Test Loss: {test_loss:.4f}")


# Create a plot to compare the forecast with the actual test data
predictions = model.predict(test_data)
plt.figure(figsize=(14,7))
plt.plot(test_output, label='Actual Data', color='orange')
plt.plot(predictions, label='Forecasted Data', color='green')
plt.title('NARX Model Evaluation')
plt.xlabel('Date')
plt.ylabel('Mean Temp')
plt.legend()
plt.show()

# Calculate the root mean squared error
mse = mean_squared_error(test_output, predictions)
rmse = mse**0.5
print('RMSE:', rmse)

#Question 3: Do the same as the previous question, but using LSTM and GRU

In [ ]:
lags = 3

df_train = pd.read_csv("/content/DailyDelhiClimateTrain.csv", index_col="date")
df_test = pd.read_csv("/content/DailyDelhiClimateTest.csv", index_col="date")


df_val = df_train[-100:]
df_train = df_train[:-100]

scaler_type=StandardScaler()
df_train = scaler_type.fit_transform(df_train)
df_val = scaler_type.transform(df_val)
df_test = scaler_type.transform(df_test)



train_data = []
train_output = []

for i in range((df_train.shape[0])-(lags)-1):
  train_data.append(df_train[i:i+lags,:])
  train_output.append(df_train[i+lags+1,0])

train_data = np.array(train_data)
train_output = np.array(train_output)


val_data = []
val_output = []

for i in range((df_val.shape[0])-(lags)-1):
  val_data.append(df_val[i:i+lags,:])
  val_output.append(df_val[i+lags+1,0])

val_data = np.array(val_data)
val_output = np.array(val_output)


test_data = []
test_output = []

for i in range((df_test.shape[0])-(lags)-1):
  test_data.append(df_test[i:i+lags,:])
  test_output.append(df_test[i+lags+1,0])

test_data = np.array(test_data)
test_output = np.array(test_output)


print(train_data.shape)
print(train_output.shape)

print(val_data.shape)
print(val_output.shape)

print(test_data.shape)
print(test_output.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import GRU
from keras import metrics
import keras
import tensorflow as tf
#import os

epochs = 100
batch_size = 8

#Early stopping is used to avoid overfitting
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', min_delta=1e-2, patience=15, verbose=0, mode='auto',
    baseline=None, restore_best_weights=True)


# define cnn model
def define_model():
  model = Sequential()
  #model.add(LSTM(12, kernel_initializer='he_uniform', return_sequences=True, input_shape=(3,4)))
  model.add(GRU(12, kernel_initializer='he_uniform', return_sequences=True, input_shape=(3,4)))
  #model.add(keras.layers.Bidirectional(LSTM(12, return_sequences=True),input_shape=(3,4)))
  model.add(Flatten())
  model.add(Dropout(0.25))
  model.add(Dense(4, activation='relu', kernel_initializer='he_uniform'))
  model.add(Dense(1))
	# compile model
  opt = tf.keras.optimizers.Adam(learning_rate=0.001)
  model.compile(optimizer=opt, loss='mean_squared_error', metrics=['mean_squared_error'])
  model.build()
  print(model.summary())
  return model


#Train network
model = define_model()
model_history = model.fit(x=train_data, y=train_output, validation_data=(val_data,val_output), epochs=epochs, batch_size=batch_size, shuffle=True, callbacks=[early_stop], verbose=1)

In [ ]:
# Create a plot to compare the forecast with the actual test data
predictions = model.predict(test_data)
plt.figure(figsize=(14,7))
plt.plot(test_output, label='Actual Data', color='orange')
plt.plot(predictions, label='Forecasted Data', color='green')
plt.title('NARX Model Evaluation')
plt.xlabel('Date')
plt.ylabel('Mean Temp')
plt.legend()
plt.show()

# Calculate the root mean squared error
mse = mean_squared_error(test_output, predictions)
rmse = mse**0.5
print('RMSE:', rmse)